# MRI brain-tumor classifier — fine-tune to lift 4-class accuracy (GPU)

**Goal.** Continue-train the platform's 4-class brain-tumor image classifier
(`Devarshi/Brain_Tumor_Classification`) so the *classifier's* test accuracy rises
from the measured baseline to the 90 %+ range you asked for.

**Baseline.** `tools/eval_mri_classifier.py`, run on the Kaggle
`brain-tumor-mri-dataset` **Testing/** split, scores the stock hub model at
**0.804 (80.4 %) accuracy**. That is the number we are trying to beat.

**Realistic target.** **0.92–0.97 (92–97 %)** test accuracy. This dataset is
clean and the task is well separated, so continue-training a model that already
sits at 80 % into the low-/mid-90s is realistic. We do **not** promise a fixed
number — paste the real printout from the last cell back and we verify it
locally.

**Honest caveat (read this for the defence).** This fine-tune validates the
**classifier only**. The MRI U-Net **segmentation** is a *different model on a
different dataset* (TCGA-LGG, which has pixel masks) and is **not** touched
here — its Dice (~0.85) is unaffected. See `CLAUDE.md` → "Things that look like
bugs but aren't". Do not conflate "classifier accuracy" with "segmentation
quality"; they are measured on different data.

**Self-contained.** This notebook needs **only** (1) the HuggingFace hub (for the
starting weights, auto-downloaded by `transformers`) and (2) Kaggle (for the
dataset). **No `medical-platform.zip` on Drive is required** — unlike the EEG/ECG
notebooks. Drive is mounted only at the very end to save the trained weights out.

**What you get back.** A `save_pretrained` directory zipped to
`My Drive/colab_pfe_outputs/mri_vit/vit_brain_tumor.zip`. Unzip it locally into
`backend/models_weights/vit_brain_tumor/` and `get_mri_classifier()` auto-detects
it (or set `VIT_BRAIN_TUMOR_WEIGHTS`). With no local dir, the platform behaves
exactly as before.

> Note on the architecture name: the hub repo is *called*
> "Brain_Tumor_Classification" and the platform refers to it as "the ViT", but the
> actual checkpoint is a **Swin-tiny** image classifier. That is why every cell
> below uses `AutoModelForImageClassification` / `AutoImageProcessor` (which load
> whatever architecture the config declares) instead of hardcoding a `ViT*` class.
> Hardcoding `ViTForImageClassification` would fail to load this checkpoint.

## 1. Confirm the GPU

*Runtime → Change runtime type → T4 GPU* first. A T4 is plenty; one fine-tune run is ~20–60 min.

In [ ]:
import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — set Runtime → GPU")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Dependencies

Colab already ships compatible `torch`, `transformers`, `scikit-learn`, `Pillow`
and `numpy` — **nothing to install**. (We download the dataset by calling the
Kaggle API directly, so the `kaggle` CLI is not needed either — the classic CLI
does not understand new-style `KGAT_` tokens.)

In [ ]:
import transformers, sklearn
print("transformers:", transformers.__version__)
print("sklearn:", sklearn.__version__)

## 3. Kaggle credentials + dataset download

The dataset is the public **`masoudnickparvar/brain-tumor-mri-dataset`** (the same
one `tools/eval_mri_classifier.py` is pointed at for the 80.4 % baseline). It is a
plain public dataset — **no competition rules to accept**.

Two ways to authenticate — the cell below supports both:
- **New-style token (recommended)** — looks like `KGAT_…` (kaggle.com →
  *Settings* → *API* → create token). Paste it when prompted; **no username, no
  kaggle.json needed**.
- **Legacy `kaggle.json`** — press Enter at the prompt instead, then upload the file.

The token is read with `getpass`, so it is never displayed or saved in the notebook.

In [ ]:
from getpass import getpass
import os

tok = getpass("Paste your Kaggle token (KGAT_...), or press Enter to upload kaggle.json instead: ").strip()
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    print("KAGGLE_API_TOKEN set (Bearer auth - no username needed).")
else:
    from google.colab import files
    import json
    up = files.upload()                         # pick your kaggle.json
    k = json.load(open("kaggle.json"))
    os.environ["KAGGLE_USERNAME"] = k["username"]
    os.environ["KAGGLE_KEY"] = k["key"]
    print("legacy kaggle.json creds set for", k["username"])

In [ ]:
import base64
import http.client
import os
import shutil
import urllib.request
import zipfile

DATA_ROOT = "/content/data"
ZIP_PATH = "/content/brain-tumor-mri-dataset.zip"
os.makedirs(DATA_ROOT, exist_ok=True)

def _auth_header():
    # Same precedence as the repo's own downloader (tools/train_eeg_head.py):
    # new-style KGAT_ Bearer token first, then legacy username+key Basic auth.
    tok = os.environ.get("KAGGLE_API_TOKEN")
    if tok:
        return "Bearer " + tok
    user, key = os.environ.get("KAGGLE_USERNAME"), os.environ.get("KAGGLE_KEY")
    if user and key:
        return "Basic " + base64.b64encode(f"{user}:{key}".encode()).decode()
    raise RuntimeError("No Kaggle credentials - run the credentials cell first.")

if not os.path.exists(ZIP_PATH):
    # Kaggle answers with a 302 to a signed storage URL; follow it WITHOUT the
    # auth header (the signed URL rejects extra Authorization headers).
    conn = http.client.HTTPSConnection("www.kaggle.com", timeout=300)
    conn.request(
        "GET",
        "/api/v1/datasets/download/masoudnickparvar/brain-tumor-mri-dataset",
        headers={"Authorization": _auth_header()},
    )
    r = conn.getresponse()
    if r.status in (301, 302, 303, 307, 308):
        url = r.getheader("Location")
        r.read()
        conn.close()
        with urllib.request.urlopen(url, timeout=600) as resp, open(ZIP_PATH, "wb") as f:
            shutil.copyfileobj(resp, f)
    elif r.status == 200:
        with open(ZIP_PATH, "wb") as f:
            shutil.copyfileobj(r, f)
        conn.close()
    else:
        body = r.read(200)
        conn.close()
        raise RuntimeError(f"Kaggle download failed: HTTP {r.status}: {body[:160]!r}")
print("zip size: %.1f MB" % (os.path.getsize(ZIP_PATH) / 1048576))

with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_ROOT)

TRAIN_DIR = os.path.join(DATA_ROOT, "Training")
TEST_DIR = os.path.join(DATA_ROOT, "Testing")
print("Training exists:", os.path.isdir(TRAIN_DIR), "->", sorted(os.listdir(TRAIN_DIR)))
print("Testing  exists:", os.path.isdir(TEST_DIR), "->", sorted(os.listdir(TEST_DIR)))

## 4. Load the starting model + processor, and pin the label mapping

We **continue training from** `Devarshi/Brain_Tumor_Classification` (not from
scratch). The class ↔ id mapping is read straight off that model's config and is
**byte-identical** to what the platform ships:

```
id2label = {0: "glioma_tumor", 1: "meningioma_tumor", 2: "no_tumor", 3: "pituitary_tumor"}
label2id = {"glioma_tumor": 0, "meningioma_tumor": 1, "no_tumor": 2, "pituitary_tumor": 3}
```

The MRI pipeline reads `model.config.id2label` for the predicted label string, so
we assert the loaded config matches this exactly and never reorder it. The Kaggle
**folder** names drop the `_tumor` suffix (`glioma`, `meningioma`, `notumor`,
`pituitary`), so we map folder → id explicitly below.

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

SOURCE = "Devarshi/Brain_Tumor_Classification"
processor = AutoImageProcessor.from_pretrained(SOURCE)
model = AutoModelForImageClassification.from_pretrained(SOURCE)
model.to(DEVICE)

# The exact mapping we MUST preserve (matches the cached config.json).
EXPECTED_ID2LABEL = {0: "glioma_tumor", 1: "meningioma_tumor", 2: "no_tumor", 3: "pituitary_tumor"}
loaded = {int(k): v for k, v in model.config.id2label.items()}
assert loaded == EXPECTED_ID2LABEL, f"id2label drift! got {loaded}"
print("architecture:", model.config.architectures, "| model_type:", model.config.model_type)
print("id2label OK:", loaded)
print("label2id OK:", model.config.label2id)
print("processor size:", processor.size, "| mean:", processor.image_mean, "| std:", processor.image_std)

# Kaggle folder name -> model class id (folders drop the '_tumor' suffix).
FOLDER_TO_ID = {"glioma": 0, "meningioma": 1, "notumor": 2, "pituitary": 3}

## 5. Preprocessing parity — IDENTICAL to eval + the deployed pipeline

This project's documented **#1 trap** is preprocessing mismatch. The deployed
classifier path (`apps/inference/mri_pipeline.py`) and the baseline harness
(`tools/eval_mri_classifier.py`) both do exactly:

```python
image_rgb = load_image_universal(path)          # PIL .open().convert("RGB") -> uint8 array
inputs = processor(images=Image.fromarray(image_rgb), return_tensors="pt")
```

i.e. **open as RGB → the HuggingFace image processor** (resize 224 bicubic +
ImageNet mean/std normalisation). We reuse that *same `processor`* for
validation and test, so our numbers are directly comparable to the 80.4 %
baseline. Training adds only light, label-preserving augmentation (horizontal
flip) **before** the identical processor — val/test get **no** augmentation.

In [ ]:
import glob, random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

def list_samples(split_dir):
    items = []
    for folder, cls_id in FOLDER_TO_ID.items():
        for ext in IMAGE_EXTS:
            for p in glob.glob(os.path.join(split_dir, folder, "*" + ext)):
                items.append((p, cls_id))
    return items

class MRIDataset(Dataset):
    def __init__(self, items, processor, train=False):
        self.items = items
        self.processor = processor
        self.train = train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label = self.items[i]
        # IDENTICAL to load_image_universal for standard images: open + RGB.
        img = Image.open(path).convert("RGB")
        if self.train and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)   # only light, label-safe aug
        pv = self.processor(images=img, return_tensors="pt")["pixel_values"][0]
        return {"pixel_values": pv, "labels": torch.tensor(label, dtype=torch.long)}

all_train = list_samples(TRAIN_DIR)
test_items = list_samples(TEST_DIR)
random.seed(0)
random.shuffle(all_train)

# Stratified-ish 15% held-out validation fraction.
VAL_FRAC = 0.15
by_cls = {}
for p, c in all_train:
    by_cls.setdefault(c, []).append((p, c))
train_items, val_items = [], []
for c, lst in by_cls.items():
    n_val = max(1, int(len(lst) * VAL_FRAC))
    val_items += lst[:n_val]
    train_items += lst[n_val:]
random.shuffle(train_items)

print(f"train: {len(train_items)}  val: {len(val_items)}  test: {len(test_items)}")
from collections import Counter
print("train class counts:", dict(Counter(c for _, c in train_items)))
print("test  class counts:", dict(Counter(c for _, c in test_items)))

BATCH = 32
train_loader = DataLoader(MRIDataset(train_items, processor, train=True),  batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(MRIDataset(val_items,   processor, train=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(MRIDataset(test_items,  processor, train=False), batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## 6. Fine-tune (continue training)

A plain PyTorch loop (more transparent than `Trainer` and free of version
quirks). Small LR because we start from an already-good model; AdamW + cosine
decay, mixed precision on GPU. We keep the checkpoint with the best **validation
accuracy**.

In [ ]:
import numpy as np

EPOCHS = 6
LR = 2e-5
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = total = 0
    preds_all, labels_all = [], []
    for batch in loader:
        pv = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        with torch.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            logits = model(pixel_values=pv).logits
        preds = logits.argmax(-1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
        preds_all.append(preds.cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    return correct / total, np.concatenate(preds_all), np.concatenate(labels_all)

best_val = -1.0
best_state = None
import copy
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for step, batch in enumerate(train_loader):
        pv = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", enabled=(DEVICE == "cuda")):
            out = model(pixel_values=pv, labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
    val_acc, _, _ = evaluate(val_loader)
    print(f"epoch {epoch}/{EPOCHS}  train_loss={running/len(train_loader):.4f}  val_acc={val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc
        best_state = copy.deepcopy(model.state_dict())

print(f"best val accuracy: {best_val:.4f}")
if best_state is not None:
    model.load_state_dict(best_state)   # restore best checkpoint before test

## 7. Evaluate on the held-out **Testing/** split

Same `processor` transform as the deployed pipeline — directly comparable to the
**0.804** baseline. Prints overall accuracy, a per-class report, and the
confusion matrix.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_acc, test_preds, test_labels = evaluate(test_loader)
BASELINE = 0.804
LABELS_ORDER = [0, 1, 2, 3]
TARGET_NAMES = [EXPECTED_ID2LABEL[i] for i in LABELS_ORDER]

print(f"TEST accuracy: {test_acc:.4f}   (baseline 0.804)")
print()
print(classification_report(test_labels, test_preds, labels=LABELS_ORDER,
                            target_names=TARGET_NAMES, digits=4))
cm = confusion_matrix(test_labels, test_preds, labels=LABELS_ORDER)
print("Confusion matrix (rows=truth, cols=pred), order:", TARGET_NAMES)
print(cm)

## 8. Save → zip → copy to Drive

`save_pretrained` writes `config.json` (with the preserved id2label),
`model.safetensors`, and `preprocessor_config.json`. We re-assert the mapping,
zip the directory, mount Drive, and copy the zip to
`My Drive/colab_pfe_outputs/mri_vit/`.

**Bring it home:** unzip into `backend/models_weights/vit_brain_tumor/` and
`get_mri_classifier()` auto-loads it (env override: `VIT_BRAIN_TUMOR_WEIGHTS`).

In [ ]:
import shutil

OUT_DIR = "/content/vit_brain_tumor"
# Re-assert and explicitly set the mapping so the saved config can never drift.
model.config.id2label = {i: EXPECTED_ID2LABEL[i] for i in LABELS_ORDER}
model.config.label2id = {v: k for k, v in EXPECTED_ID2LABEL.items()}
model.save_pretrained(OUT_DIR)
processor.save_pretrained(OUT_DIR)
print("saved files:", sorted(os.listdir(OUT_DIR)))

ZIP_PATH = "/content/vit_brain_tumor.zip"
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)
shutil.make_archive("/content/vit_brain_tumor", "zip", OUT_DIR)

from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/My Drive/colab_pfe_outputs/mri_vit"
os.makedirs(DRIVE_DIR, exist_ok=True)
DRIVE_ZIP = os.path.join(DRIVE_DIR, "vit_brain_tumor.zip")
shutil.copy(ZIP_PATH, DRIVE_ZIP)
print("copied to:", DRIVE_ZIP)

## 9. Results — paste this whole block back to Claude

In [ ]:
print("=== RESULTS — paste this back to Claude ===")
print(f"dataset            : masoudnickparvar/brain-tumor-mri-dataset (Kaggle)")
print(f"start model        : {SOURCE}  ({model.config.model_type})")
print(f"train / val / test : {len(train_items)} / {len(val_items)} / {len(test_items)}")
print(f"epochs / lr / batch: {EPOCHS} / {LR} / {BATCH}")
print(f"id2label (preserved): {{0:'glioma_tumor',1:'meningioma_tumor',2:'no_tumor',3:'pituitary_tumor'}}")
print(f"BASELINE test acc  : 0.8040")
print(f"FINE-TUNED test acc: {test_acc:.4f}")
print(f"best val acc       : {best_val:.4f}")
print(f"delta vs baseline  : {test_acc - 0.804:+.4f}")
print("confusion matrix (rows=truth, cols=pred), order glioma/meningioma/notumor/pituitary:")
print(cm)
print(f"Drive output zip   : {DRIVE_ZIP}")
print(f"-> unzip locally to: backend/models_weights/vit_brain_tumor/  (get_mri_classifier auto-detects it)")
print("=== END RESULTS ===")